In [ ]:
from libraries import *
from parameters import *
from util import *

from statsmodels.stats.multitest import multipletests


In [ ]:
adata = sc.read_h5ad("./../../Data/ComboScreen.h5ad")


In [ ]:
adata = adata[adata.obs[['ASCL1', 'KLF14', 'NEUROD1',
       'NEUROG1', 'NR3C1', 'NTC', 'SIM1', 'TET2', 'TWIST1', 'VSX1', 'ZNF385A',
       'ZNF547', 'ZNF660', 'ZNF776']].sum(axis=1) < 3]

In [ ]:
cols = ['ASCL1', 'KLF14', 'NEUROD1', 'NEUROG1', 'NR3C1', 'NTC', 'SIM1',
        'TET2', 'TWIST1', 'VSX1', 'ZNF385A', 'ZNF547', 'ZNF660', 'ZNF776']

def combine_onehot(row):
    # Select all column names where value == 1
    active = [col for col in cols if row[col] == 1]
    # Join multiple actives with '+', or return 'None' if none are active
    return '+'.join(active) if active else 'None'

adata.obs['perturbation'] = adata.obs[cols].apply(combine_onehot, axis=1)


In [ ]:
sc.pp.normalize_total(adata, target_sum=20000)
sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, n_top_genes=3000)
# sc.pp.scale(adata, max_value=9)
# sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
# sc.pp.neighbors(adata, 
#                 n_neighbors=8,
#                 metric=par_downstream_neighbor_metric,
#                 n_pcs=50)
# sc.tl.umap(adata)

In [ ]:
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str)
)

In [ ]:
doxo1Signature = pd.read_csv("Doxo_1_differentiated.DEGs.csv", index_col=0)

In [ ]:
sc.tl.score_genes(
        adata,
        gene_list=list(doxo1Signature.names),
        score_name="Doxo1program_score",
        use_raw=False
    )


In [ ]:
adata_differentiated=adata[adata.obs["final_label"].isin(["Differentiated_1", "Differentiated-2"]),:]

In [ ]:
# sc.pl.umap(adata, color='Doxo1program_score', 
#        #legend_loc='on data', 
#        legend_fontoutline=3, 
#        legend_fontsize=14, 
#        legend_fontweight='normal', 
#        show=False, 
#        size=0.3)

In [ ]:
adata_day04 = adata[adata.obs["time_point"] == "day04",]
adata_day10 = adata[adata.obs["time_point"] == "day10",]

adata_differentiated_day04 = adata_differentiated[adata_differentiated.obs["time_point"] == "day04",]
adata_differentiated_day10 = adata_differentiated[adata_differentiated.obs["time_point"] == "day10",]

In [ ]:
k=adata_differentiated.obs["perturbation_time"].value_counts()
k["NTC_day04"]

In [ ]:
from scipy.stats import ranksums

def score_and_wilcoxon_vs_baseline(
    adata,
    group_col="perturbation_time",
    baseline_group="NTC_day04",
    score_name="program_score",
    rankby_abs=False,
    out_csv="program_score_wilcoxon_vs_NTC_day04.csv",
):
    """
    1) Scores cells by a gene-set signature.
    2) For each group in group_col, tests program_score vs baseline_group using Wilcoxon rank-sum (two-sided).
    3) Saves p-values + mean differences (group - baseline).
    """

    # ---- 0) Ensure we have baseline
    if baseline_group not in adata.obs[group_col].unique():
        raise ValueError(f"baseline_group='{baseline_group}' not found in adata.obs['{group_col}'].")

    # ---- 1) Prepare gene list (intersection with data)

    scores = adata.obs[score_name].to_numpy()
    groups = adata.obs[group_col].astype(str)

    base_mask = (groups == baseline_group)
    base_scores = scores[base_mask]

    results = []
    for g in sorted(groups.unique()):
        mask = (groups == g)
        g_scores = scores[mask]

        # Skip if empty (shouldn't happen) or baseline itself
        if g_scores.size == 0:
            continue

        # Wilcoxon rank-sum (Mann–Whitney style). SciPy's ranksums is unpaired, two-sided.
        # If you want Mann–Whitney U instead, tell me.
        stat, pval = ranksums(g_scores, base_scores)

        results.append({
            group_col: g,
            "n_group": int(g_scores.size),
            "n_baseline": int(base_scores.size),
            "mean_group": float(np.mean(g_scores)),
            "mean_baseline": float(np.mean(base_scores)),
            "mean_diff_group_minus_baseline": float(np.mean(g_scores) - np.mean(base_scores)),
            "wilcoxon_stat": float(stat),
            "pval": float(pval),
        })

    res = pd.DataFrame(results)
    res["FDR"] = multipletests(res["pval"],method="fdr_bh")[1]

    # Optional sorting
    if rankby_abs:
        res["abs_mean_diff"] = res["mean_diff_group_minus_baseline"].abs()
        res = res.sort_values(["abs_mean_diff", "pval"], ascending=[False, True]).drop(columns=["abs_mean_diff"])
    else:
        res = res.sort_values(["pval", "mean_diff_group_minus_baseline"], ascending=[True, False])

    # ---- 4) Save
    res.to_csv(out_csv, index=False)

    return res



In [ ]:
res_dif_day04 = score_and_wilcoxon_vs_baseline(
    adata_differentiated_day04,
    group_col="perturbation_time",
    baseline_group="NTC_day04",
    score_name="Doxo1program_score",
    rankby_abs=False,
    out_csv="Doxo1program_score_wilcoxon_vs_NTC_day04_differentiated.csv",
)

In [ ]:
res_dif_day10 = score_and_wilcoxon_vs_baseline(
    adata_differentiated_day10,
    group_col="perturbation_time",
    baseline_group="NTC_day10",
    score_name="Doxo1program_score",
    rankby_abs=False,
    out_csv="Doxo1program_score_wilcoxon_vs_NTC_day10_differentiated.csv",
)

In [ ]:
res_day04 = score_and_wilcoxon_vs_baseline(
    adata_day04,
    group_col="perturbation_time",
    baseline_group="NTC_day04",
    score_name="Doxo1program_score",
    rankby_abs=False,
    out_csv="Doxo1program_score_wilcoxon_vs_NTC_day04.csv",
)

In [ ]:
res_day10 = score_and_wilcoxon_vs_baseline(
    adata_day10,
    group_col="perturbation_time",
    baseline_group="NTC_day10",
    score_name="Doxo1program_score",
    rankby_abs=False,
    out_csv="Doxo1program_score_wilcoxon_vs_NTC_day10.csv",
)

In [ ]:
res_day04.loc[(res_day04.pval < 0.05) & (res_day04.mean_diff_group_minus_baseline > 0),:]

In [ ]:
res_dif_day04.loc[res_dif_day04.pval < 0.05,:]

In [ ]:
res_day10.loc[(res_day10.pval < 0.05) & (res_day10.mean_diff_group_minus_baseline > 0),:]

In [ ]:
res_dif_day10.loc[(res_dif_day10.pval < 0.05) & (res_dif_day10.mean_diff_group_minus_baseline > 0),:]

In [ ]:
from scipy.stats import fisher_exact

def getEnrichmentResults(inadata):
    
    ct = pd.crosstab(
    inadata.obs["final_label"],
    inadata.obs["perturbation_time"])
    
    results = []

    total_cells = ct.values.sum()

    for label in ct.index:
        for pert in ct.columns:
            # 2x2 table
            a = ct.loc[label, pert]                      # in label & pert
            b = ct.loc[label].sum() - a                  # in label, not pert
            c = ct[pert].sum() - a                       # in pert, not label
            d = total_cells - (a + b + c)                # neither

            table = [[a, b],
                     [c, d]]

            oddsratio, pval = fisher_exact(table, alternative="two-sided")

            results.append({
                "final_label": label,
                "Perturbation_time": pert,
                "n_cells": a,
                "odds_ratio": oddsratio,
                "pval": pval
            })

    enrichment_df = pd.DataFrame(results)
    enrichment_df["qval"] = multipletests(enrichment_df["pval"],method="fdr_bh")[1]
    enrichment_df = enrichment_df.loc[enrichment_df.qval < 0.1,:]
    return enrichment_df

In [ ]:
enrichmentDay04=getEnrichmentResults(inadata=adata_day04)
enrichmentDay04

In [ ]:
enrichmentDay10=getEnrichmentResults(inadata=adata_day10)
enrichmentDay10

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Extract the relevant columns
df = adata_differentiated.obs[["perturbation_time", "Doxo1program_score"]].copy()

# Compute median per group and get ordering
order = (
    df.groupby("perturbation_time")["Doxo1program_score"]
      .median()
      .sort_values(ascending=False)
      .index
)

# Plot
plt.figure(figsize=(28, 5))

sns.violinplot(
    data=df,
    x="perturbation_time",
    y="Doxo1program_score",
    order=order,
    inner="box",     # shows median + IQR (ggplot-like)
    cut=0,
    scale="width"
)

plt.xticks(rotation=90)
plt.xlabel("Perturbation + time")
plt.ylabel("Doxo1 program score")
plt.title("Doxo1 program score by perturbation_time (ordered by median)")

plt.tight_layout()
plt.show()


In [ ]:
sc.pl.umap(adata_day10, color='Doxo1program_score', 
       #legend_loc='on data', 
       legend_fontoutline=3, 
       legend_fontsize=14, 
       legend_fontweight='normal', 
       show=False, 
       size=0.3)